**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/05-resnet-from-paper.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/resnet-residual-block) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7E · ResNet from the Paper (He et al., 2016)

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 60–75 min (experiment ≈ 1 min on CPU) | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | the residual block, ResNet-18/34/50 with exactly torchvision's parameter counts, a check with real ImageNet weights, and the paper's Figure 1 experiment reproduced on your laptop |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

📄 **Paper:** K. He, X. Zhang, S. Ren, J. Sun, *Deep Residual Learning for Image Recognition*, CVPR 2016 (Best Paper) ·
[arXiv:1512.03385](https://arxiv.org/abs/1512.03385) · [PDF](https://arxiv.org/pdf/1512.03385) · read **§1 (Figure 1),
§3.1–3.3 (Figure 2, Table 1)**. One of the most cited papers in all of science.

> *"When deeper networks are able to start converging, a degradation problem has been exposed: with the network depth
> increasing, accuracy gets saturated … and then degrades rapidly. Unexpectedly, such degradation is not caused by
> overfitting, and adding more layers to a suitably deep model leads to higher training error."* (§1)

**The idea in one line:** instead of asking a block to learn $H(x)$, let it learn the **change** $F(x) = H(x) - x$ and
output $F(x) + x$. If a layer is not useful it can simply learn $F(x) = 0$ and pass $x$ through untouched.

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## 1 · The basic residual block (Figure 2, Table 1 "[3×3, 64; 3×3, 64]")

```
x ──► conv3×3 ─► BN ─► ReLU ─► conv3×3 ─► BN ──►(+)──► ReLU ──► out
│                                              ▲
└──────────────── shortcut ────────────────────┘   identity, or 1×1 conv + BN if the shape changes
```
Details from §3.3–3.4 that you must not miss:
* convolutions have **no bias** (`bias=False`): the BatchNorm right after has its own shift, so a bias is wasted;
* the first conv of a block may have **stride 2** (halves the size); then the shortcut must also change shape:
  a **1 × 1 conv with stride 2 + BN** ("projection shortcut", option B in the paper);
* the last ReLU comes **after** the addition.

### 🖐 Your turn 1: `BasicBlock(cin, cout, stride)`
Keep the attribute names (`conv1, bn1, conv2, bn2, relu, downsample`): they are torchvision's names, so later we can load real ImageNet weights into **your** model.
<details><summary><b>Hint 1</b></summary>

`self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)`; `self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)`.

</details>
<details><summary><b>Hint 2</b></summary>

`if stride != 1 or cin != cout: self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))`

</details>
<details><summary><b>Hint 3</b></summary>

forward: `identity = x if self.downsample is None else self.downsample(x)`; …; `return self.relu(out + identity)`.

</details>

### 📄 ↔ 💻 Paper ↔ Code: Figure 2, Eq. (1) and Eq. (2) → `BasicBlock`

<table>
<tr><th width="50%" align="left">📄 Paper · He et al. 2016, §3.2, Figure 2, Eq. (1)–(2)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_resnet_fig2.png" width="100%" alt="p_resnet_fig2.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_eq1.png" width="100%" alt="pc_r_eq1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_eq2.png" width="100%" alt="pc_r_eq2.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class BasicBlock(nn.Module):
    # F(x, {W_i}): conv3x3 -&gt; BN -&gt; ReLU -&gt; conv3x3 -&gt; BN   (&quot;weight layer&quot; boxes)
    # shortcut: identity x            Eq. (1), when the shapes match
    #           W_s x = 1x1 conv + BN Eq. (2), when stride != 1 or cin != cout
    def forward(self, x):
        # out = F(x)
        # return relu(out + shortcut(x))       the (+) and the last &quot;relu&quot; of Fig. 2</pre>
<b>Shapes:</b> BasicBlock(64, 128, stride=2): (B, 64, 56, 56) → (B, 128, 28, 28); the shortcut needs W<sub>s</sub> (1×1 conv, stride 2)
</td></tr>
</table>

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1                    # output channels = cout * expansion

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        # TODO: conv1, bn1, conv2, bn2 (convs without bias)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        # TODO: projection shortcut when the shape changes

    def forward(self, x):
        # TODO
        return x

In [ ]:
C.check_basic_block(BasicBlock)

## 2 · The whole network (Table 1)

| stage | output (224 input) | ResNet-18 | ResNet-34 | ResNet-50 |
|---|---|---|---|---|
| conv1 | 112 × 112 | 7 × 7, 64, stride 2 | same | same |
| conv2_x | 56 × 56 | 3 × 3 max pool, stride 2, then **2** blocks | **3** blocks | **3** bottlenecks |
| conv3_x | 28 × 28 | **2** blocks, 128 | **4** | **4** |
| conv4_x | 14 × 14 | **2** blocks, 256 | **6** | **6** |
| conv5_x | 7 × 7 | **2** blocks, 512 | **3** | **3** |
| | 1 × 1 | average pool, 1000-d fc | same | same |

`Bottleneck` (for ResNet-50/101/152) is given below: 1 × 1 (reduce) → 3 × 3 → 1 × 1 (expand × 4).

### 🖐 Your turn 2: `_make_layer`: one stage = `n` blocks, only the first one may have stride 2
<details><summary><b>Hint 1</b></summary>

`layers = [block(self.cin, width, stride)]`, then `self.cin = width * block.expansion`.

</details>
<details><summary><b>Hint 2</b></summary>

Then `n - 1` more blocks with stride 1: `block(self.cin, width, 1)`. Return `nn.Sequential(*layers)`.

</details>

### 📄 ↔ 💻 Paper ↔ Code: Table 1 → `ResNet(block, layers)`

<table>
<tr><th width="50%" align="left">📄 Paper · He et al. 2016, §3.4, Table 1, Figure 5</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_table1.png" width="100%" alt="pc_r_table1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_fig5.png" width="100%" alt="pc_r_fig5.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># Table 1, column &quot;18-layer&quot;:
#   conv1    7x7, 64, stride 2   -&gt; 3x3 max pool, stride 2       112 -&gt; 56
#   conv2_x  [3x3, 64;  3x3, 64]  x 2    56 x 56   first block stride 1
#   conv3_x  [3x3, 128; 3x3, 128] x 2    28 x 28   first block stride 2
#   conv4_x  [3x3, 256; 3x3, 256] x 2    14 x 14   first block stride 2
#   conv5_x  [3x3, 512; 3x3, 512] x 2     7 x 7    first block stride 2
#   average pool, 1000-d fc
resnet18 = ResNet(BasicBlock, [2, 2, 2, 2])
resnet34 = ResNet(BasicBlock, [3, 4, 6, 3])
resnet50 = ResNet(Bottleneck, [3, 4, 6, 3])   # Fig. 5 right: 1x1, 3x3, 1x1 (x4)</pre>
</td></tr>
</table>

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, cin, width, stride=1):
        super().__init__()
        cout = width * self.expansion
        self.conv1 = nn.Conv2d(cin, width, 1, bias=False); self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, 3, stride, 1, bias=False); self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or cin != cout:
            self.downsample = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return self.relu(out + identity)

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()
        self.cin = 64
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(block, 64, layers[0], 1)
        self.layer2 = self._make_layer(block, 128, layers[1], 2)
        self.layer3 = self._make_layer(block, 256, layers[2], 2)
        self.layer4 = self._make_layer(block, 512, layers[3], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, width, n, stride):
        # TODO
        return nn.Sequential()

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1))

resnet18 = lambda: ResNet(BasicBlock, [2, 2, 2, 2])
resnet34 = lambda: ResNet(BasicBlock, [3, 4, 6, 3])
resnet50 = lambda: ResNet(Bottleneck, [3, 4, 6, 3])

In [ ]:
C.check_resnet(resnet18(), 18)
C.check_resnet(resnet34(), 34)
C.check_resnet(resnet50(), 50)

## 3 · The ultimate test: load the **real** ImageNet weights into your code

If your model is *exactly* the paper's network, the weights trained by the PyTorch team must fit into it with
`strict=True` (every name and shape must match) and it must recognise real photos. (Needs internet: ≈ 45 MB download.)

In [ ]:
import torchvision
from torchvision.models import ResNet18_Weights
try:
    weights = ResNet18_Weights.IMAGENET1K_V1
    mine = resnet18()
    mine.load_state_dict(torchvision.models.resnet18(weights=weights).state_dict(), strict=True)
    mine.eval()
    from sklearn.datasets import load_sample_image          # two photos bundled with scikit-learn
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    for a, name in zip(ax, ["china.jpg", "flower.jpg"]):
        photo = load_sample_image(name)
        x = weights.transforms()(torch.from_numpy(photo).permute(2, 0, 1)).unsqueeze(0)
        with torch.no_grad():
            p = mine(x).softmax(1)[0]
        top = p.topk(3)
        label = "\n".join(f"{weights.meta['categories'][i]}: {v:.0%}" for v, i in zip(top.values, top.indices))
        a.imshow(photo); a.set_xticks([]); a.set_yticks([]); a.set_xlabel(label, fontsize=10)
    plt.suptitle("YOUR ResNet-18 code + the official ImageNet weights: top-3 guesses"); plt.tight_layout(); plt.show()
    print("✅ strict=True loading worked: your network is exactly torchvision's ResNet-18.")
except Exception as e:
    print("Skipped (no internet or a mismatch):", type(e).__name__, str(e)[:300])

Notice the flower: ImageNet's 1000 classes include "daisy" but not "dahlia", so the network picks the closest texture it
knows. A model can only name what was in its training labels, which is also a paper-reading lesson: check the **dataset**.

## 4 · Reproduce Figure 1: deeper plain networks get **worse**

The paper's Figure 1 shows a 56-layer "plain" network with a **higher training error** than a 20-layer one. We repeat
the experiment in miniature on scikit-learn's 8 × 8 handwritten digits (1,797 images, no download):

* **plain-8, plain-20, plain-56**: a stack of conv-BN-ReLU blocks, *no* shortcut;
* **ResNet-56**: exactly the same layers *with* shortcuts.

Same data, same optimiser (SGD + momentum, as in the paper), same 15 epochs. ≈ 1 minute on a CPU.

### 📄 ↔ 💻 Paper ↔ Code: Figure 1: the degradation problem you are about to reproduce

<table>
<tr><th width="50%" align="left">📄 Paper · He et al. 2016, §1, Figure 1; §4.2, Table 6</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_resnet_fig1.png" width="100%" alt="p_resnet_fig1.png"><br></td>
<td width="50%" valign="top">
<b>📄 The paper reports</b>
<ul>
<li>Figure 1: on CIFAR-10 a <b>56-layer plain</b> network has <b>higher training and test error</b> than a 20-layer one, so this is not overfitting.</li>
<li>With shortcuts the trend reverses: ResNet-56 6.97 % and ResNet-110 6.43 % test error on CIFAR-10 (Table 6).</li>
<li>ImageNet: an ensemble of ResNets won ILSVRC 2015 with 3.57 % top-5 error.</li>
</ul>
<b>💻 You measure in this notebook</b>
<ul>
<li>plain-20 vs plain-56 vs ResNet-56 on the same data, same steps</li>
<li>if your plain-56 is worse than your plain-20, you have reproduced the paper's central observation</li>
</ul>
</td></tr>
</table>

In [ ]:
Xtr, ytr, Xte, yte = C.load_digits_tensors()
print("train", tuple(Xtr.shape), "· test", tuple(Xte.shape))

class SmallBlock(nn.Module):
    def __init__(self, c, residual):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(c, c, 3, 1, 1, bias=False); self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = self.bn2(self.conv2(torch.relu(self.bn1(self.conv1(x)))))
        return torch.relu(out + x) if self.residual else torch.relu(out)

class SmallNet(nn.Module):
    """depth = 1 stem conv + 2 per block + 1 fc  ->  n_blocks = 3, 9, 27 gives 8, 20, 56 layers"""
    def __init__(self, n_blocks, residual, c=16):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, c, 3, 1, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU())
        self.blocks = nn.Sequential(*[SmallBlock(c, residual) for _ in range(n_blocks)])
        self.fc = nn.Linear(c, 10)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean((2, 3)))

def train_small(n_blocks, residual, epochs=15, lr=0.05):
    C.seed_everything(0)
    m = SmallNet(n_blocks, residual).to(device)
    opt = torch.optim.SGD(m.parameters(), lr, momentum=0.9, weight_decay=1e-4)
    hist = []
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xtr)); total = 0.0
        for i in range(0, len(Xtr), 64):
            idx = perm[i:i + 64]
            loss = F.cross_entropy(m(Xtr[idx].to(device)), ytr[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        hist.append(total / len(Xtr))
    m.eval()
    with torch.no_grad():
        acc = (m(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
    return hist, acc

results = {}
for name, nb, res in [("plain-8", 3, False), ("plain-20", 9, False), ("plain-56", 27, False), ("ResNet-56", 27, True)]:
    t0 = time.time()
    results[name] = train_small(nb, res)
    print(f"{name:10s} final training loss {results[name][0][-1]:.3f} · test accuracy {results[name][1]:.1%} · {time.time() - t0:.0f} s")

In [ ]:
plt.figure(figsize=(6, 3.5))
for name, style in [("plain-8", "C0--"), ("plain-20", "C1--"), ("plain-56", "C3--"), ("ResNet-56", "C2-")]:
    plt.plot(range(1, 16), results[name][0], style, lw=2, label=f"{name} (test {results[name][1]:.0%})")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend()
plt.title("Deeper plain nets train WORSE; shortcuts fix it"); plt.tight_layout(); plt.show()

**Read the plot like the paper's Figure 1:** the curves are *training* loss, so this is **not overfitting**: plain-56
cannot even fit the training data, although it could in theory copy plain-20 and set its extra layers to identity. With
shortcuts, identity is the *easiest* thing to learn, and ResNet-56 trains as well as the shallow network.

## ✍️ Your turn
1. Count the layers: why does `n_blocks = 27` give 56 layers?
2. Run plain-56 again with `lr=0.01`. Does a smaller learning rate rescue it? What does that tell you?
3. Remove the BatchNorms from `SmallBlock` (replace with `nn.Identity()`) and compare plain-20 vs ResNet-20.
4. How many parameters does ResNet-152 have? Build it from Table 1 (`Bottleneck`, `[3, 8, 36, 3]`) and compare with `torchvision.models.resnet152()`.
5. ⭐ Read §4.2 of the paper ("Exploring over 1000 layers"): what happened with 1202 layers, and why?

**Next:** [Lab 7F · Vision Transformer from the paper](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/06-vit-from-paper.ipynb)